#Tests new transcript output against a ground truth

In [1]:
import pandas as pd 
import numpy as np
import os 

In [5]:

# Correct transcript
test_path = 'C://Users//pisces2//Documents//Audio_Transcription_Deidentification//updated_speakers.xlsx'

# Wisper transcript
train_path = 'C://Users//pisces2//Documents//Audio_Transcription_Deidentification//transcripts//hamlet_test_chunkworks_2024.10.24.csv'


df_test = pd.read_excel(test_path)


df_train = pd.read_csv(train_path)





In [22]:
# Get and match speakers- are they the same?

df_train = df_train[['Speaker', 'Transcription']]

df_train = df_train.reset_index(drop=True)
df_teest = df_test.reset_index(drop=True)

#collapse speakers

new_train = []

current_speaker = df_train.iloc[0]['Speaker']
current_text = df_train.iloc[0]['Transcription']

for i in range(1, len(df_train)):
    # If speakers are the same 
    if df_train.iloc[i]['Speaker'] == current_speaker:
        #add text together
        #print("current text", current_text)
        #print("current line", df_train.iloc[i]['Transcription'])
        current_text = current_text + ' ' + df_train.iloc[i]['Transcription']
    else:
        # add to record and start again
        new_train.append({'Speaker': current_speaker, 'Transcription': current_text})
        current_speaker = df_train.iloc[i]['Speaker']
        current_text = df_train.iloc[i]['Transcription'] if pd.notna(df_train.iloc[i]['Transcription']) else ''

new_train.append({'Speaker': current_speaker, 'Transcription': current_text})

df_train = pd.DataFrame(new_train)





# df_test.columns = df_train.columns
# df_test.index = df_train.index

# print(df_train.columns)
# print(df_test.columns)
# print(df_train.shape)
# print(df_train)

# print("test")
# print(df_test)
# print(df_test.shape)

# comparison = df_test == df_train
# print(comparison)

# differences = df_test[~comparison.all(axis=1)]

# print("Rows with differences")
# print(differences)

Compare text : all text or by row.

In [91]:
import string
def compare_text(test_set, train_set, by_row):

    test_words = test_set.translate(str.maketrans('', '', string.punctuation))
    train_words = train_set.translate(str.maketrans('', '', string.punctuation))

    test_words = test_words.lower()
    train_words = train_words.lower()

    test_w = test_words.split()
    train_w = train_words.split()
 

    i = 0
    j = 0
    errors = 0
    matches = 0
    #print(train_w, test_w)
    while i < len(train_w) and j < len(test_w):


        print(train_w[i], test_w[j])
        if train_w[i] == test_w[j]:
            print("match")
            i += 1
            j += 1
            #if by_row:
               # errors = 0
            matches += 1
            #print(matches)
        else:

            errors += 1
            j += 1

    print("matches", matches, "errors", errors, "len test", len(test_w), "len_train", len(train_w))
    return errors, len(test_w)

In [93]:
# Combine text 
import string
all_train_text = ""
all_test_text = ""

for row in range(len(df_train)):

    if pd.notna(df_train.iloc[row]['Transcription']):
        all_train_text = all_train_text + df_train.iloc[row]['Transcription']

for row in range(len(df_test)):

    if pd.notna(df_test.iloc[row]['Transcription']):
        all_test_text = all_test_text + df_test.iloc[row]['Transcription']


# errors, total_words = compare_text(all_test_text, all_train_text, False) 
# print(errors, total_words)
error_results  = []
#train
i = 0
#test
j = 0

df_train_cleaned = df_train.dropna(subset=['Transcription'], how='all')
df_train_cleaned = df_train_cleaned[df_train_cleaned['Transcription'].str.strip() != '']
for index in range(len(df_train_cleaned)):
    if index == 2:
        break

    train_text = df_train_cleaned.iloc[i]['Transcription']
    test_text = df_test.loc[j, 'Transcription']

    print("trying texts")
    print("train", train_text)
    print("text: ", test_text)
    errors, num_words = compare_text(test_text, train_text, by_row = True)
    print(errors, num_words)
    error_results.append({
        'speaker_id': df_train_cleaned.iloc[i]['Speaker'],
        'errors': errors/num_words
    })
    # If the entire row is bad, whisper missed it most likely. try next test row.
    if errors/num_words == 1.0: 

        j += 1
    else:
        i += 1
        j += 1

    print("new ij ", i, j )



print(error_results)

trying texts
train  to him, tell him his pranks have been too broad to bear with, and your grace has screened and stood between much heat and him. I'll silence me even here. Are you be rowd with him?
text:  He will come straight. Look you lay home to him. Tell him his pranks have been too broad to bear  with. And that your Grace hath screened and stood between.  Much heat and him. I’ll silence me even here.  Pray you, be round with him.
to he
to will
to come
to straight
to look
to you
to lay
to home
to to
match
him him
match
tell tell
match
him him
match
his his
match
pranks pranks
match
have have
match
been been
match
too too
match
broad broad
match
to to
match
bear bear
match
with with
match
and and
match
your that
your your
match
grace grace
match
has hath
has screened
has and
has stood
has between
has much
has heat
has and
has him
has i’ll
has silence
has me
has even
has here
has pray
has you
has be
has round
has with
has him
matches 16 errors 29 len test 45 len_train 36
29 45
new 